# L0 Dataset Reduction to 50% with Optuna Optimization

This notebook uses Optuna to automatically find the best L0 regularization parameters for reducing the Enhanced CPS dataset to exactly 50% of its original size while maintaining high calibration accuracy.

## Setup and Imports

In [1]:
# Required packages: uv pip install microcalibrate optuna

import numpy as np
import pandas as pd
import warnings
import logging
from pathlib import Path
import optuna
from optuna.samplers import TPESampler

# Import Hugging Face Hub for dataset download
from huggingface_hub import hf_hub_download

# Configure logging
logging.getLogger('microcalibrate').setLevel(logging.WARNING)
logging.getLogger('optuna').setLevel(logging.WARNING)
warnings.filterwarnings('ignore')

# Import microcalibrate modules
import sys
sys.path.append('../src')
from microcalibrate.calibration import Calibration

# Import PolicyEngine US data modules
sys.path.append('../../policyengine-us-data')
sys.path.append('../../policyengine-us')

from policyengine_us_data.datasets import EnhancedCPS_2024
from policyengine_us_data.utils import build_loss_matrix
from policyengine_us import Microsimulation
from policyengine_core.data import Dataset

print("✓ All modules imported successfully")

# Set random seed for reproducibility
np.random.seed(42)
optuna.logging.set_verbosity(optuna.logging.WARNING)

/Users/elenacura/Desktop/PolicyEngine/policyengine-us-data/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ All modules imported successfully


## Load Enhanced CPS Dataset

In [2]:
def load_dataset_for_optuna():
    """Load and prepare dataset for Optuna optimization."""
    
    def get_data_directory():
        data_dir = Path("../data/input/enhanced_cps")
        data_dir.mkdir(parents=True, exist_ok=True)
        return data_dir
    
    def get_enhanced_cps_dataset(time_period=2024):
        print("Downloading Enhanced CPS dataset...")
        dataset_path = hf_hub_download(
            repo_id="policyengine/policyengine-us-data",
            filename="enhanced_cps_2024.h5",
            local_dir=get_data_directory(),
        )
        return Dataset.from_file(dataset_path, time_period=time_period)
    
    # Load dataset
    ecps_dataset = get_enhanced_cps_dataset()
    sim = Microsimulation(dataset=ecps_dataset)
    weights = sim.calculate("household_weight", 2024).values
    
    # Build targets
    loss_matrix, targets_array = build_loss_matrix(EnhancedCPS_2024, 2024)
    
    # Clean targets
    problematic_targets = [
        "nation/irs/adjusted gross income/total/AGI in 10k-15k/taxable/Head of Household",
        "nation/irs/adjusted gross income/total/AGI in 15k-20k/taxable/Head of Household", 
        "nation/irs/adjusted gross income/total/AGI in 10k-15k/taxable/Married Filing Jointly/Surviving Spouse",
        "nation/irs/adjusted gross income/total/AGI in 15k-20k/taxable/Married Filing Jointly/Surviving Spouse",
        "nation/irs/count/count/AGI in 10k-15k/taxable/Head of Household",
        "nation/irs/count/count/AGI in 15k-20k/taxable/Head of Household",
        "nation/irs/count/count/AGI in 10k-15k/taxable/Married Filing Jointly/Surviving Spouse", 
        "nation/irs/count/count/AGI in 15k-20k/taxable/Married Filing Jointly/Surviving Spouse",
        "state/RI/adjusted_gross_income/amount/-inf_1",
        "nation/irs/exempt interest/count/AGI in -inf-inf/taxable/All",
    ]
    
    zero_mask = np.isclose(targets_array, 0.0, atol=0.1)
    problematic_mask = loss_matrix.columns.isin(problematic_targets)
    keep_mask = ~(zero_mask | problematic_mask)
    keep_indices = np.where(keep_mask)[0]
    
    loss_matrix_clean = loss_matrix.iloc[:, keep_indices]
    targets_clean = targets_array[keep_indices]
    target_names_clean = loss_matrix_clean.columns.values
    
    # Align dimensions
    min_size = min(len(weights), len(loss_matrix_clean))
    if len(loss_matrix_clean) > min_size:
        loss_matrix_clean = loss_matrix_clean.iloc[:min_size]
    if len(weights) > min_size:
        weights = weights[:min_size]
    
    print(f"✅ Dataset loaded: {len(weights):,} households, {len(targets_clean)} targets")
    print(f"📊 Target: Reduce to ~{len(weights)//2:,} households (50%)")
    
    return weights, loss_matrix_clean, targets_clean, target_names_clean

# Load dataset
weights, estimate_matrix, targets, target_names = load_dataset_for_optuna()

INFO:root:Targeting Medicaid enrollment for AK with target 231577k
INFO:root:Targeting Medicaid enrollment for AL with target 766009k
INFO:root:Targeting Medicaid enrollment for AR with target 733561k
INFO:root:Targeting Medicaid enrollment for AZ with target 1778734k
INFO:root:Targeting Medicaid enrollment for CA with target 12172695k
INFO:root:Targeting Medicaid enrollment for CO with target 1058326k
INFO:root:Targeting Medicaid enrollment for CT with target 904321k
INFO:root:Targeting Medicaid enrollment for DC with target 240020k
INFO:root:Targeting Medicaid enrollment for DE with target 236840k
INFO:root:Targeting Medicaid enrollment for FL with target 3568648k
INFO:root:Targeting Medicaid enrollment for GA with target 1699279k
INFO:root:Targeting Medicaid enrollment for HI with target 376318k
INFO:root:Targeting Medicaid enrollment for IA with target 586748k
INFO:root:Targeting Medicaid enrollment for ID with target 296968k
INFO:root:Targeting Medicaid enrollment for IL with targ

✅ Dataset loaded: 41,310 households, 2801 targets
📊 Target: Reduce to ~20,655 households (50%)


## Optuna 50% Optimization Framework

In [3]:
def objective_50_percent(trial):
    """
    Optuna objective function specifically for 50% dataset reduction.
    Returns a score where LOWER = BETTER.
    """
    # Suggest parameters optimized for 50% target
    l0_lambda = trial.suggest_float('l0_lambda', 1e-6, 2e-5, log=True)
    temperature = trial.suggest_float('temperature', 1.0, 3.0)
    init_mean = trial.suggest_float('init_mean', 0.995, 0.9999)
    epochs = trial.suggest_int('epochs', 1500, 2500, step=250)
    
    try:
        # Run L0 calibration
        calibrator = Calibration(
            weights=weights,
            targets=targets,
            target_names=target_names,
            estimate_matrix=estimate_matrix,
            regularize_with_l0=True,
            l0_lambda=l0_lambda,
            temperature=temperature,
            init_mean=init_mean,
            epochs=epochs,
            learning_rate=1e-3,
            noise_level=10.0
        )
        
        # Run calibration (suppress output for cleaner Optuna logs)
        performance_df = calibrator.calibrate()
        sparse_weights = calibrator.sparse_weights
        
        # Calculate metrics
        original_size = len(weights)
        sparse_size = np.sum(sparse_weights > 0)
        size_reduction_pct = (original_size - sparse_size) / original_size * 100
        
        # Calculate accuracy
        sparse_estimates = sparse_weights @ estimate_matrix.values
        sparse_errors = sparse_estimates - targets
        targets_safe = np.where(np.abs(targets) < 1e-6, 1e-6, targets)
        sparse_rel_abs_errors = np.abs(sparse_errors) / np.abs(targets_safe)
        sparse_mrae = np.mean(sparse_rel_abs_errors) * 100
        sparse_within_10pct = np.mean(sparse_rel_abs_errors <= 0.1) * 100
        
        # 50% target objective (minimize deviation from 50% + maintain accuracy)
        target_size_reduction = 50.0
        size_deviation = abs(size_reduction_pct - target_size_reduction)
        
        # Penalty for size deviation (exponential for large deviations)
        if size_deviation <= 2.0:  # 2% tolerance
            size_penalty = 0
        else:
            size_penalty = ((size_deviation - 2.0) / 10.0) ** 2
        
        # Penalty for poor accuracy
        if sparse_within_10pct < 85:  # Minimum 85% accuracy
            accuracy_penalty = (85 - sparse_within_10pct) / 100
        else:
            accuracy_penalty = 0
        
        # Combined objective (lower = better)
        objective_score = (
            0.6 * size_penalty +           # 60% weight on hitting 50% target
            0.3 * (sparse_mrae / 100) +    # 30% weight on accuracy (MRAE)
            0.1 * accuracy_penalty         # 10% penalty for low accuracy
        )
        
        # Log intermediate results for monitoring
        trial.set_user_attr('size_reduction_pct', size_reduction_pct)
        trial.set_user_attr('sparse_mrae', sparse_mrae)
        trial.set_user_attr('sparse_within_10pct', sparse_within_10pct)
        trial.set_user_attr('size_deviation', size_deviation)
        
        return objective_score
        
    except Exception as e:
        # Return high penalty for failed trials
        trial.set_user_attr('error', str(e))
        return 1000.0

def run_final_calibration(best_params):
    """Run final calibration with best parameters and save results."""
    print(f"\n🔄 Running final calibration with best parameters...")
    
    calibrator = Calibration(
        weights=weights,
        targets=targets,
        target_names=target_names,
        estimate_matrix=estimate_matrix,
        regularize_with_l0=True,
        l0_lambda=best_params['l0_lambda'],
        temperature=best_params['temperature'],
        init_mean=best_params['init_mean'],
        epochs=best_params['epochs'],
        learning_rate=1e-3,
        noise_level=10.0
    )
    
    performance_df = calibrator.calibrate()
    sparse_weights = calibrator.sparse_weights
    
    # Calculate final metrics
    original_size = len(weights)
    sparse_size = np.sum(sparse_weights > 0)
    size_reduction_pct = (original_size - sparse_size) / original_size * 100
    
    sparse_estimates = sparse_weights @ estimate_matrix.values
    sparse_errors = sparse_estimates - targets
    targets_safe = np.where(np.abs(targets) < 1e-6, 1e-6, targets)
    sparse_rel_abs_errors = np.abs(sparse_errors) / np.abs(targets_safe)
    sparse_mrae = np.mean(sparse_rel_abs_errors) * 100
    sparse_within_10pct = np.mean(sparse_rel_abs_errors <= 0.1) * 100
    
    size_deviation = abs(size_reduction_pct - 50.0)
    
    # Save calibration log
    save_calibration_log(calibrator, best_params, size_reduction_pct, sparse_mrae, sparse_within_10pct)
    
    return {
        'original_size': original_size,
        'sparse_size': sparse_size,
        'size_reduction_pct': size_reduction_pct,
        'size_deviation': size_deviation,
        'sparse_mrae': sparse_mrae,
        'sparse_within_10pct': sparse_within_10pct,
        'calibrator': calibrator
    }

def save_calibration_log(calibrator, params, size_reduction_pct, sparse_mrae, sparse_within_10pct):
    """Save calibration log in dashboard format."""
    output_dir = Path('../results')
    output_dir.mkdir(exist_ok=True)
    
    try:
        sparse_weights = calibrator.sparse_weights
        estimates = sparse_weights @ estimate_matrix.values
        
        calibration_logs = []
        for i, target_name in enumerate(target_names):
            if i >= len(targets) or i >= len(estimates):
                continue
                
            target_val = targets[i]
            estimate_val = estimates[i]
            error = estimate_val - target_val
            abs_error = abs(error)
            rel_abs_error = abs_error / abs(target_val) if abs(target_val) > 1e-6 else 0
            
            log_entry = {
                'config_id': f"optuna_50pct_λ={params['l0_lambda']:.0e}_T={params['temperature']:.2f}",
                'epoch': params['epochs'] - 1,
                'loss': sparse_mrae / 100,
                'target_name': target_name,
                'target': target_val,
                'estimate': estimate_val,
                'error': error,
                'abs_error': abs_error,
                'rel_abs_error': rel_abs_error,
                'l0_lambda': params['l0_lambda'],
                'temperature': params['temperature'],
                'init_mean': params['init_mean'],
                'size_reduction_pct': size_reduction_pct,
                'sparse_within_10pct': sparse_within_10pct
            }
            calibration_logs.append(log_entry)
        
        if calibration_logs:
            log_df = pd.DataFrame(calibration_logs)
            log_path = output_dir / 'calibration_log.csv'
            log_df.to_csv(log_path, index=False)
            print(f"✅ Calibration log saved: {log_path}")
            print(f"   • {len(log_df):,} entries for {len(target_names)} targets")
            
    except Exception as e:
        print(f"❌ Could not save calibration log: {e}")

print("Optuna 50% optimization framework ready!")

Optuna 50% optimization framework ready!


## Run Optuna Optimization for 50% Target

In [ ]:
print("🎯 STARTING OPTUNA OPTIMIZATION FOR 50% DATASET REDUCTION")
print("=" * 60)

# Create Optuna study
study = optuna.create_study(
    direction='minimize',  # Minimize the objective score
    sampler=TPESampler(seed=42),  # Tree-structured Parzen Estimator
    study_name='l0_50_percent_optimization'
)

# Set optimization parameters
n_trials = 30  # Number of parameter combinations to try
print(f"🔍 Running {n_trials} optimization trials...")
print(f"📊 Target: Find parameters that achieve exactly 50% size reduction")
print(f"🎯 Objective: Minimize deviation from 50% while maintaining accuracy")

# Run optimization
study.optimize(objective_50_percent, n_trials=n_trials, show_progress_bar=True)

# Get best results
best_trial = study.best_trial
best_params = best_trial.params
best_score = best_trial.value

print(f"\n🏆 OPTUNA OPTIMIZATION COMPLETE!")
print(f"=" * 50)
print(f"Best objective score: {best_score:.4f}")
print(f"\n📋 BEST PARAMETERS:")
for param, value in best_params.items():
    print(f"  {param}: {value}")

print(f"\n📊 BEST TRIAL RESULTS:")
print(f"  Size reduction: {best_trial.user_attrs['size_reduction_pct']:.1f}%")
print(f"  Deviation from 50%: {best_trial.user_attrs['size_deviation']:.1f}%")
print(f"  Accuracy (within 10%): {best_trial.user_attrs['sparse_within_10pct']:.1f}%")
print(f"  Mean relative error: {best_trial.user_attrs['sparse_mrae']:.2f}%")

# Show optimization history
print(f"\n📈 OPTIMIZATION PROGRESS (Top 5 trials):")
print(f"{'Trial':<6} {'Score':<8} {'Size %':<8} {'Dev %':<8} {'Acc %':<8} {'λ':<10} {'T':<6}")
print("-" * 60)

for i, trial in enumerate(sorted(study.trials, key=lambda x: x.value)[:5]):
    if hasattr(trial, 'user_attrs') and 'size_reduction_pct' in trial.user_attrs:
        print(f"{trial.number:<6} {trial.value:<8.4f} {trial.user_attrs['size_reduction_pct']:<8.1f} "
              f"{trial.user_attrs['size_deviation']:<8.1f} {trial.user_attrs['sparse_within_10pct']:<8.1f} "
              f"{trial.params['l0_lambda']:<10.0e} {trial.params['temperature']:<6.2f}")

🎯 STARTING OPTUNA OPTIMIZATION FOR 50% DATASET REDUCTION
🔍 Running 30 optimization trials...
📊 Target: Find parameters that achieve exactly 50% size reduction
🎯 Objective: Minimize deviation from 50% while maintaining accuracy


Sparse reweighting progress: 100%|██████████| 4000/4000 [08:15<00:00,  8.07epoch/s, loss=0.12, loss_rel_change=-0.705]


## Final Calibration with Best Parameters

In [ ]:
# Run final calibration with best parameters
final_results = run_final_calibration(best_params)

print("\n" + "=" * 60)
print("✅ FINAL 50% DATASET REDUCTION RESULTS")
print("=" * 60)

print(f"\n📊 FINAL METRICS:")
print(f"  Dataset size: {final_results['original_size']:,} → {final_results['sparse_size']:,} households")
print(f"  Size reduction: {final_results['size_reduction_pct']:.1f}%")
print(f"  Deviation from 50% target: {final_results['size_deviation']:.1f}%")
print(f"  Calibration accuracy: {final_results['sparse_within_10pct']:.1f}% targets within 10%")
print(f"  Mean relative error: {final_results['sparse_mrae']:.2f}%")

print(f"\n🎛️ OPTIMAL PARAMETERS FOUND BY OPTUNA:")
for param, value in best_params.items():
    print(f"  {param}: {value}")

# Quality assessment
print(f"\n🔍 QUALITY ASSESSMENT:")
if final_results['size_deviation'] <= 3.0:
    print("  ✅ Excellent size targeting: Within 3% of 50% target")
elif final_results['size_deviation'] <= 5.0:
    print("  ⚠️  Good size targeting: Within 5% of 50% target") 
else:
    print("  ❌ Poor size targeting: >5% deviation from 50% target")
    
if final_results['sparse_within_10pct'] >= 90:
    print("  ✅ Excellent accuracy: >90% targets within 10%")
elif final_results['sparse_within_10pct'] >= 85:
    print("  ⚠️  Good accuracy: 85-90% targets within 10%") 
else:
    print("  ❌ Poor accuracy: <85% targets within 10%")

print(f"\n💾 OUTPUT FILES:")
print(f"  • calibration_log.csv: Saved with optimal 50% reduction results")
print(f"  • Ready for dashboard analysis")

print(f"\n✨ Optuna automatically found the best parameters for 50% reduction!")
print(f"   No manual parameter tuning required.")

## Optuna Study Analysis (Optional)

In [ ]:
# Optional: Analyze the optimization study
print("📊 OPTUNA STUDY ANALYSIS")
print("=" * 40)

# Parameter importance
try:
    importance = optuna.importance.get_param_importances(study)
    print(f"\n🔍 PARAMETER IMPORTANCE:")
    for param, imp in importance.items():
        print(f"  {param}: {imp:.3f}")
except:
    print("\n🔍 Parameter importance analysis requires more trials")

# Trial statistics
successful_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
failed_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.FAIL]

print(f"\n📈 TRIAL STATISTICS:")
print(f"  Total trials: {len(study.trials)}")
print(f"  Successful: {len(successful_trials)}")
print(f"  Failed: {len(failed_trials)}")
print(f"  Success rate: {len(successful_trials)/len(study.trials)*100:.1f}%")

if successful_trials:
    scores = [t.value for t in successful_trials]
    print(f"\n🎯 OBJECTIVE SCORE STATISTICS:")
    print(f"  Best (lowest): {min(scores):.4f}")
    print(f"  Worst (highest): {max(scores):.4f}")
    print(f"  Mean: {np.mean(scores):.4f}")
    print(f"  Std: {np.std(scores):.4f}")

    # Best trials achieving close to 50%
    close_to_50_trials = [
        t for t in successful_trials 
        if hasattr(t, 'user_attrs') and 'size_deviation' in t.user_attrs 
        and t.user_attrs['size_deviation'] <= 5.0
    ]
    
    print(f"\n🎯 TRIALS WITHIN 5% OF 50% TARGET: {len(close_to_50_trials)}/{len(successful_trials)}")
    
print(f"\n✅ Optuna optimization analysis complete!")
print(f"   Best parameters are ready for production use.")